In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
import pandas as pd
aus_retail = pd.read_csv('data/aus_retail.csv', sep=',', na_values=[''], quotechar='"',parse_dates=['Month'])
aus_retail.head(5)

In [ ]:
aus_retail = (
    aus_retail
    .loc[lambda x: x["Industry"] == "Newspaper and book retailing"]
    .assign(Year = aus_retail['Month'].dt.year.astype(str))
    .groupby("Year", as_index=False)
    .agg({"Turnover":"sum"})
)
aus_retail

In [ ]:
aus_retail = pd.read_csv("data/aus_retail.csv", parse_dates=["Month"])
print_retail = (
    aus_retail
    .loc[lambda x: x["Industry"] == "Newspaper and book retailing"]
    .assign(ds=lambda x: x["Month"].dt.year)
    .groupby("ds", as_index=False)
    ["Turnover"].sum()
)
print_retail


In [ ]:
fig1, ax1 = plt.subplots()
fig2, ax2 = plt.subplots()
ax1.plot(print_retail["ds"],print_retail['Turnover'])
global_economy = pd.read_csv('data/global_economy.csv', sep=',', na_values=[''], quotechar='"')
aus_economy = global_economy.loc[lambda x: x["unique_id"] == "Australia"]
df = (
    aus_economy.merge(print_retail, on="ds", how="left")
    .dropna()
    .assign(adjusted_turnover=lambda x: x["Turnover"]/x["CPI"] * 100)
)

ax2.plot(df["ds"],df['adjusted_turnover'])
plt.show()

In [ ]:
import pandas as pd
global_economy = pd.read_csv('data/global_economy.csv', sep=',', na_values=[''], quotechar='"')
aus_economy = global_economy.loc[lambda x: x["unique_id"] == "Australia"]
df = (
    aus_economy.merge(print_retail, on="ds", how="left")
    .dropna()
    .assign(adjusted_turnover=lambda x: x["Turnover"]/x["CPI"] * 100)
)
plt.figure(2)
plt.plot(df["ds"],df['adjusted_turnover'])
plt.show()


In [ ]:
import pandas as pd
aus_production = pd.read_csv('data/aus_production.csv', sep=',', na_values=[''], quotechar='"')
aus_production.head(5)

In [ ]:
import pandas as pd
us_employment = pd.read_csv('data/us_employment.csv', sep=',', parse_dates=["ds"],  na_values=[''], quotechar='"')
us_retail_employment = us_employment.loc[lambda x: (x["unique_id"] == "Retail Trade")&(x["ds"].dt.year >=1990)]
us_retail_employment = us_retail_employment.assign(
    Year = us_retail_employment["ds"].dt.year
)

fig3,ax3 = plt.subplots()
ax3.plot(us_retail_employment["Year"],us_retail_employment['y'])
plt.show()



In [ ]:
us_retail_employment

In [ ]:
us_employment = pd.read_csv("data/us_employment.csv", parse_dates=["ds"])
us_retail_employment = us_employment.loc[lambda x:
    (x["unique_id"] == "Retail Trade") & (x["ds"] >= "1990")
]
from fpppy.utils import plot_series
plot_series(us_retail_employment,
    xlabel="Month [1M]", ylabel="Persons (thousands)",
    title="Total employment in US retail")
us_retail_employment

In [ ]:
from statsmodels.tsa.seasonal import STL
stl = STL(us_retail_employment["y"], period=12)
res = stl.fit()
dcmp = pd.DataFrame({
    "ds": us_retail_employment["ds"],
    "data": us_retail_employment["y"],
    "trend": res.trend,
    "seasonal": res.seasonal,
    "remainder": res.resid,
}).reset_index(drop=True)
dcmp.head()


In [ ]:
fig, ax = plt.subplots()
sns.lineplot(data=dcmp, x="ds", y="data", color="gray", label="Data")
sns.lineplot(data=dcmp, x="ds", y="trend", color="#D55E00", label="Trend")
ax.set(
    title="Total employment in US retail",
    xlabel="Month [1M]",
    ylabel="Persons (thousands)",
)
plt.show()

In [ ]:
aus_exports = (
    global_economy
    .loc[lambda x: x["unique_id"] == "Australia"]
    .assign(MA_5=lambda x: x["Exports"].rolling(5, center=True).mean())
    .assign(MA_7=lambda x: x["Exports"].rolling(7, center=True).mean())
)
aus_exports

In [ ]:
fig1, ax1 = plt.subplots()
fig2, ax2 = plt.subplots()

ax1.plot(aus_exports["ds"],aus_exports["MA_5"],color="#D55E00", label="MA_5")
ax2.plot(aus_exports["ds"],aus_exports["MA_7"],color="#D55E00", label="MA_7")
ax1.set(title="MA_5")
ax1.plot(aus_exports["ds"],aus_exports["Exports"],color="black", label="Exports")
ax2.plot(aus_exports["ds"],aus_exports["Exports"],color="black", label="Exports")
ax2.set(title="MA_7")
plt.show()

In [ ]:
us_employment = pd.read_csv('data/us_employment.csv', sep=',', parse_dates=["ds"],  na_values=[''], quotechar='"')
us_retail_employment = us_employment.loc[lambda x: (x["unique_id"] == "Retail Trade")&(x["ds"].dt.year >=1990)]
us_retail_employment = us_retail_employment.assign(
    Year = us_retail_employment["ds"].dt.year
)
us_retail_employment.head()


In [ ]:
us_retail_employment["12-MA"] = us_retail_employment["y"].rolling(window=12, center=True).mean()
us_retail_employment["2*12-MA"] = us_retail_employment["12-MA"].rolling(window=2, center=True).mean()
us_retail_employment.dropna()
fig,ax4 = plt.subplots()
ax4.plot(us_retail_employment["ds"],us_retail_employment["y"],color="gray", label="12-MA")
ax4.plot(us_retail_employment["ds"],us_retail_employment["2*12-MA"],color="#D55E00", label="2*12-MA")
plt.legend(loc="upper left")
ax4.set(title="12-MA")
plt.show()

In [ ]:
stl = STL(us_retail_employment["y"], period=12,seasonal=13,trend=21,robust=True)
res = stl.fit()
dcmp = pd.DataFrame({
    "ds": us_retail_employment["ds"],
    "data": us_retail_employment["y"],
    "trend": res.trend,
    "seasonal": res.seasonal,
    "remainder": res.resid,
})
fig,axes = plt.subplots(4,1,figsize=(12,8))
axes[0].plot(dcmp["ds"],dcmp["data"],color="gray", label="Data")
axes[1].plot(dcmp["ds"],dcmp["trend"],color="pink", label="Data")
axes[2].plot(dcmp["ds"],dcmp["seasonal"],color="orange", label="Data")
axes[3].plot(dcmp["ds"],dcmp["remainder"],color="red", label="Data")
plt.legend(loc="upper left")
plt.show()